In [ ]:
import pandas as pd
import json
import time
from openai import OpenAI
from tqdm import tqdm

In [ ]:
# Initialize OpenAI Client
client = OpenAI(api_key="UvvltYhVUCvfEGkdP2w2wzmKXUAQ")
# 1. Load your datasets
train_df = pd.read_csv('public/train.csv')
test_df = pd.read_csv('public/test.csv')
model_name = "gpt-4.1-mini"
ai_client = client

In [ ]:
def create_anchor_library(df):
    """Creates a dictionary of reference code snippets for each smell type."""
    library = {}
    smell_types = df['smell_type'].unique()
    
    for smell in smell_types:
        subset = df[df['smell_type'] == smell]
        
        # Select one clear '2' (Low) and one clear '4' (High) as anchors
        ref_2 = subset[subset['severity_score'] == 2].iloc[0]['code_snippet'] if not subset[subset['severity_score'] == 2].empty else "N/A"
        ref_4 = subset[subset['severity_score'] == 4].iloc[0]['code_snippet'] if not subset[subset['severity_score'] == 4].empty else "N/A"
        
        library[smell] = {"low": ref_2, "high": ref_4}
    return library

anchor_lib = create_anchor_library(train_df)

In [ ]:
# --- 2. THE SCORING FUNCTION ---
def get_llm_score(row, anchors):
    smell = row['smell_type']
    code = row['code_snippet']
    ref_low = anchors[smell]['low']
    ref_high = anchors[smell]['high']

    system_prompt = "You are a Senior Software Engineer specializing in Technical Debt and Code Quality Audits."
    
    user_prompt = f"""
    Task: Rate the severity of the code smell '{smell}' on a scale of 1 to 5.
    
    ### ANCHOR EXAMPLES FOR CALIBRATION:
    - REFERENCE (Score 2 - Minor): 
    {ref_low}
    
    - REFERENCE (Score 4 - Serious): 
    {ref_high}
    
    ### TARGET CODE TO EVALUATE:
    {code}
    
    ### SCORING RUBRIC:
    1: Negligible. Better/cleaner than the Score 2 reference.
    2: Minor. Similar impact to the Score 2 reference.
    3: Moderate. Worse than Score 2, but not as risky as Score 4.
    4: High. Similar impact/risk to the Score 4 reference.
    5: Critical. Significantly more dangerous or complex than the Score 4 reference.
    
    Provide your analysis and score in JSON format.
    """

    try:
        response = ai_client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0  # Deterministic output
        )
        
        data = json.loads(response.choices[0].message.content)
        # Ensure result is an integer within 1-5
        score = int(data.get("severity_score", 3))
        return max(1, min(5, score))
    
    except Exception as e:
        print(f"Error on ID {row['id']}: {e}")
        return 3 # Safe middle-ground fallback

In [ ]:
# --- 3. EXECUTION LOOP ---
results = []
print(f"Processing {len(test_df)} test samples...")

# Using tqdm for a progress bar
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    score = get_llm_score(row, anchor_lib)
    results.append({"id": row['id'], "severity_score": score})
    
    # Optional: Small sleep to avoid rate limits if using a Tier 1 API key
    # time.sleep(0.1)

In [ ]:
# --- 4. EXPORT & VERIFICATION ---
submission_df = pd.DataFrame(results)

# CRITICAL: Verify distribution to ensure we aren't getting all 3s again
print("\n--- Prediction Distribution ---")
print(submission_df['severity_score'].value_counts().sort_index())

submission_df.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' created successfully.")